# Free boundary

:::{admonition} Experimental Feature
:class: caution
This feature is experimental and still being tested.
:::

In this notebook we cover how to setup and run a GVEC free-boundary equilibirum.

The main additional ingredient that we need is a `CoilSet`.
Here, we will use a simple example of a two field-period rotating ellipse with a dense set of coils to avoid significant coil ripple effects.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "2"
import gvec

In [ ]:
coil_set = gvec.CoilSet.load(
    "ellipstell.coils",
)

fig_coils = coil_set.plot(
    line={"color": "black"},
    showlegend=False,
)
fig_coils["layout"]["scene"]["aspectmode"] = "data"

### Free boundary  for a rotating ellipse

Given your `CoilSet`, the next step is to setup a parameter file with the initial guess for your equilibrium boundary. Here, this guess is a circular boundary with approximately the right major and minor radius.

To be able to compare our solution to the coil-field later on, we opt to solve for the vacuum case; $p(\rho)=0$, $I_\text{tor}=0$

Note, that many of the parameters which are typically used in fixed-boundary have to be set as well. This is, because in-between free-boundary updates GVEC solves for the plasma field using its fixed-boundary algorithm.

In [ ]:
# =========== SETTING UP THE PARAMETERS ===========
params = {"nfp": 2}

# PHYSICS PARAMETERS
# ==================
# total enclosed toroidal flux (scales magnetic field stength)
params["ProjectName"] = "FreeBoundaryExample"
params["PhiEdge"] = -0.3

# toroidal current profile
params["I_tor"] = {
    "type": "polynomial",  # c_0 + c_1*s + c_2*s^2 ..., s=rho^2
    "coefs": [0.0],  # [c_0,c_1]
}
# initial guess for iota
params["iota"] = {
    "type": "polynomial",
    "coefs": [0.0],
}

# pressure profile
params["pres"] = {"type": "polynomial", "coefs": [0.0]}

# BOUNDARY INITIAL GUESS
# =====================
# set hmap to cylinder coordindates: X1=R,X2=Z,zeta=-phi
params["which_hmap"] = 1
params["init_average_axis"] = True

# boundary modes
params["X1_b_cos"] = {
    (0, 0): 4.0,
    (1, 0): 0.5,
}
params["X2_b_sin"] = {
    (1, 0): 0.5,
}

# RESOLUTION & RUNTIME LIMIT
# ==========================
# maximum Fourier mode numbers for X1,X2,LA [m:poloidal,n:toroidal]
params["X1_mn_max"] = [1, 1]
params["X2_mn_max"] = [1, 1]
params["LA_mn_max"] = [1, 1]

# radial resolution parameters
# number of radial B-spline elements
params["sgrid_nElems"] = 8
# degree of B-splines for X1 and X2
params["X1X2_deg"] = 5
# degree of B-splines for LA
params["LA_deg"] = 5

# minimizer parameters
# maximum number of iterations
params["totalIter"] = int(1e6)
# stopping tolerance in the energy minimization
params["minimize_tol"] = 1e-5

params["logIter"] = 100

### Free boundary parameters

Additionally to the parameters listed above, we have to specify the section for the free-boundary algorithm titled `free_boundary`. In this section, various aspects of the free-boundary updates can be specified. However, the only required argument is `updates`, which specifies the maximum number of free-boundary updates.

We also specify some more parameters that typically might be set, such as the free-boundary minimizer. Per default this minimizer is just gradient descent (`minimizer_type = "GD"`), but a variety of other minimizer that might converge faster are available (e.g. `"BFGS"`, accelerated  GD `"aGD"`, momentum GD `"mGD"`, `"ADAM"`, $\ldots$). Here we will use a quasi-newton method, damped BFGS (`"dBFGS"`), which works well for many cases.

In [ ]:
# =========== FREE-BOUNDARY PARAMETERS ===========
params["free_boundary"] = dict(
    # REQUIRED
    updates=200,  # maximum number of free boundary updates
    # OPTIONAL BUT MIGHT BE CHANGED
    tolerance=1e-3,  # free boundary tolerance
    dt=1e-4,  # initial stepsize
    minimizer_type="dBFGS",  # which free boundary minimizer to use
    force_ratio_limit=0.01,  # termination criterion due to resolution issues (e.g. just 1% of the forces are resolved with given resolution)
)

### Angular refinement

One approach that can be quite beneficial in free boundary calculations is to ramp up the (angular) resolution. This can be done via staging in GVEC (see [stages section](/user/stages)([&#x1F310;](https://gvec.readthedocs.io/latest/user/stages.html))).

In this case we start quite simple with just $m_\text{max}=n_\text{max}=1$ modes. For this resolution we iterate on the free boundary problem until less than 5% of the forces on the boundary are representable with the given resolution. This is reflected in the `force_ratio_limit` parameter. After that, we increase the angular resolution to $m_\text{max}=n_\text{max}=2$ modes and iterate until our free boundary `tolerance` of $10^{-3}$ is reached.

In [ ]:
params["stages"] = [
    {"free_boundary": dict(force_ratio_limit=0.05)},
    {
        "free_boundary": dict(dt=0.5),
        "X1_mn_max": [2, 2],
        "X2_mn_max": [2, 2],
        "LA_mn_max": [2, 2],
    },
]

## Running free boundary

With this setup completed we can now invoke the `gvec.run` command. 

To be able to compare our initial an final solution we also create a `State` object that represents our initial guess for the equilibrium.

In [ ]:
state_initial = gvec.State.new(params)

To turn on the free boundary mode, we have to provide the `coil_set` parameter to `gvec.run`. Alternatively we could also specify a `coil_set` in the `free_boundary` section of the parameter file.
The run below can take up to two minutes to complete. Hence, we set the `loglevel` to `"INFO"`, which provides a more verbose output. 

In [ ]:
# =========== RUNNING FREE-BOUNDARY ===========
fb_run = gvec.run(
    params,
    runpath="run_free_boundary",
    coil_set=coil_set,
    loglevel="INFO",
)

## Run diagnostics

To quickly inspect the evolution of some quantities of interest with the free boundary iterations, one can use the `plot_diagnostics_minimization` method.

In [ ]:
fb_run.plot_diagnostics_minimization();

Similarly, some information on the interior intermediate fixed-boundary iterations can be obtained via the `plot_diagnostics_fixed_boundary` method.

In [ ]:
fb_run.plot_diagnostics_fixed_boundary();

For manual plotting or additional information the two diagnostics `diagnostics_stage` and `diagnostics_minimizer` can be inspected.


In [ ]:
fb_run.diagnostics_stage

In [ ]:
fb_run.diagnostics_minimizer

Now we can extract the final `State` of the run to perform some postprocessing and visualization.

In [ ]:
state_final = fb_run.state

## Postprocessing: comparison to the vacuum geometry

Since we specified zero pressure and toroidal current, we expect to recover the coil-field with GVECs free boundary solution.

To visualize this we perform fieldline tracing and overlap Poincaré sections with our initial and final flux surfaces.

In [ ]:
from gvec.coils import trace_fieldlines
import numpy as np
import matplotlib.pyplot as plt

os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
# frame aligned fieldline tracing
rho = np.linspace(1e-3, 1.0, 4)
zetas = np.linspace(0.0, np.pi / state_final.nfp, 4)
t = 800
starts = state_final.evaluate(
    "pos",
    rho=rho,
    zeta=zetas[0],
    theta=0.0,
).pos

planes = state_final.get_intersection_planes(zetas=zetas)
fieldlines = trace_fieldlines(
    starts=starts,
    coils=coil_set,
    time=t,
    events=planes,
    rtol=1e-6,
)

In [ ]:
# get the initial and final flux surfaces
thetas = np.linspace(0, 2 * np.pi, 128)
ev_initial = state_initial.evaluate(
    "X1", "X2", zeta=zetas, rho=rho, theta=thetas
).squeeze()
ev_final = state_final.evaluate(
    "X1", "X2", zeta=zetas, rho=rho, theta=thetas
).squeeze()

In [ ]:
# Plot Poincaré together with GVEC initial and final surface
fig, axs = plt.subplots(2, 2, figsize=(6, 6), tight_layout=True)
conts = []
plot_zetas = []
plot_rhos = []
legend_pad = ["_" for _ in range(len(rho) - 1)]

fieldlines_kwargs = {
    "marker": ".",
    "color": "k",
    "s": 100,
    "zorder": -1e6,
    "alpha": 0.4,
}

# visualize flux surfaces
for ax, zeta in zip(axs.flatten(), zetas):
    ax.plot(
        ev_initial.sel(zeta=zeta).X1.T,
        ev_initial.sel(zeta=zeta).X2.T,
        "C0",
        label=["initial"] + legend_pad,
    )
    ax.plot(
        ev_final.sel(zeta=zeta).X1.T,
        ev_final.sel(zeta=zeta).X2.T,
        "r",
        label=["final"] + legend_pad,
    )

# visualize Poincaré
for fieldline in fieldlines:
    ds = fieldlines[fieldline]
    for i_zeta, ax in enumerate(axs.flatten()):
        ax.scatter(
            ds[f"event_{i_zeta}_X1"],
            ds[f"event_{i_zeta}_X2"],
            **fieldlines_kwargs,
        )

# labels and legend
for ax in axs.flatten():
    ax.axis("equal")
    ax.set(xlabel=r"$X^1$", ylabel=r"$X^2$")
axs[0, 0].legend()

As we can see, the final solution is in reasonable agreement with the Poincaré sections and vastly improves upon the poor initial guess. For better agreement, one would have to iterate to lower tolerances and possibly higher resolutions.